# Experiment 14: SciBERT Embeddings + Author H-Index

**Objective**: Push F1 beyond the 63.33% ceiling by trying two untested approaches:

1. **SciBERT abstract embeddings** — Replace sparse TF-IDF (5k bag-of-words) with dense 768-dim contextual embeddings from `allenai/scibert_scivocab_uncased`. The EDA notebook (53) identified SciBERT as the most likely path to 75%+ F1.
2. **Authors H-Index** — The `Authors H-index` column exists in `cleaned_data.pkl` but was **never included in the feature matrix**. Author reputation is a strong predictor of future citation impact.

**Strategy**:
- Config A: Venue + Author (including H-index) features only (no text)
- Config B: SciBERT embeddings + Venue + Author features
- Config C: SciBERT embeddings only (ablation)
- Baseline: Current best (TF-IDF 5k + Venue + Author, no H-index)

**Train/Test split**: 2010–2017 train, 2018–2020 test (same expanded split as Exp 10c which gave 63.33%)

In [ ]:
import sys
sys.path.append('../../')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, classification_report
import lightgbm as lgb
import pickle

RANDOM_STATE = 42
print('Libraries loaded')

## 1. Load Data

In [ ]:
data_dir = Path('../../data')
feature_dir = data_dir / 'features'

# Raw cleaned data (has Authors H-index column)
df = pd.read_pickle(data_dir / 'processed/cleaned_data.pkl')

# Existing feature matrix and targets
X_all = pd.read_pickle(feature_dir / 'X_all.pkl')
y_cls = pd.read_pickle(feature_dir / 'y_classification.pkl')
metadata = pd.read_pickle(feature_dir / 'metadata.pkl')

print(f'Cleaned data: {df.shape}')
print(f'Features (X_all): {X_all.shape}')
print(f'Target: {y_cls.shape}')
print(f'\nClass distribution: {y_cls.value_counts().to_dict()}')
print(f'High-impact rate: {y_cls.mean():.1%}')

## 2. Temporal Split (2010–2017 train, 2018–2020 test)

In [ ]:
train_years = list(range(2010, 2018))
test_years  = [2018, 2019, 2020]

train_mask = df['Year'].isin(train_years)
test_mask  = df['Year'].isin(test_years)

train_idx = df[train_mask].index.intersection(X_all.index)
test_idx  = df[test_mask].index.intersection(X_all.index)

# AUB-only (institution filter if column exists)
inst_col = next((c for c in df.columns if 'institution' in c.lower() and 'num' not in c.lower()), None)
if inst_col:
    aub_mask = df[inst_col].str.contains('AUB|American University of Beirut', case=False, na=False)
    train_idx = df[train_mask & aub_mask].index.intersection(X_all.index)
    test_idx  = df[test_mask & aub_mask].index.intersection(X_all.index)
    print(f'Using AUB-only filter (col: {inst_col})')
else:
    print('No institution column found — using all papers')

y_train = y_cls.loc[train_idx]
y_test  = y_cls.loc[test_idx]

print(f'\nTrain: {len(train_idx)} papers ({y_train.mean():.1%} high-impact)')
print(f'Test:  {len(test_idx)} papers ({y_test.mean():.1%} high-impact)')

## 3. Authors H-Index Feature

`Authors H-index` in Scopus data is typically a semicolon-separated list of h-indexes, one per author.
We extract: max h-index, mean h-index, total h-index, and count of "star" authors (h ≥ 20).

In [ ]:
# Find the Authors H-index column
hindex_col = next((c for c in df.columns if 'h-index' in c.lower() or 'hindex' in c.lower()), None)
print(f'H-index column found: {hindex_col}')

if hindex_col:
    print(f'Sample values:')
    print(df[hindex_col].dropna().head(10).tolist())
    print(f'Missing: {df[hindex_col].isna().sum()} / {len(df)}')
else:
    print('Available columns containing index/impact:')
    for c in df.columns:
        if any(x in c.lower() for x in ['h-index', 'hindex', 'impact', 'citation', 'author']):
            print(f'  {c}')

In [ ]:
def parse_hindex_series(series):
    """Parse semicolon-separated h-index values into structured features."""
    results = []
    for val in series:
        if pd.isna(val) or str(val).strip() == '':
            results.append({'h_max': np.nan, 'h_mean': np.nan, 'h_sum': np.nan, 'h_star_count': np.nan})
            continue
        # Parse: could be "12;5;8" or "12 | 5 | 8" or just "12"
        parts = str(val).replace('|', ';').replace(',', ';').split(';')
        nums = []
        for p in parts:
            try:
                n = float(p.strip())
                if n >= 0:
                    nums.append(n)
            except ValueError:
                pass
        if nums:
            results.append({
                'h_max': max(nums),
                'h_mean': np.mean(nums),
                'h_sum': sum(nums),
                'h_star_count': sum(1 for h in nums if h >= 20)
            })
        else:
            results.append({'h_max': np.nan, 'h_mean': np.nan, 'h_sum': np.nan, 'h_star_count': np.nan})
    return pd.DataFrame(results, index=series.index)


if hindex_col:
    hindex_features = parse_hindex_series(df[hindex_col])
    # Impute missing with median
    for col in hindex_features.columns:
        med = hindex_features[col].median()
        hindex_features[col] = hindex_features[col].fillna(med)

    print('H-index features summary:')
    print(hindex_features.describe().round(2))
    
    # Quick check: do high-impact papers have higher h-index?
    all_idx = train_idx.union(test_idx)
    hi_mask = y_cls.loc[all_idx] == 1
    print(f'\nMean h_max — High-impact: {hindex_features.loc[all_idx][hi_mask]["h_max"].mean():.1f}')
    print(f'Mean h_max — Low-impact:  {hindex_features.loc[all_idx][~hi_mask]["h_max"].mean():.1f}')
else:
    hindex_features = pd.DataFrame(index=df.index)
    print('WARNING: No h-index column found — skipping h-index features')

## 4. Build Baseline Feature Sets

Separate structured features (venue + author) from TF-IDF text features.

In [ ]:
# Separate TF-IDF from structured features
tfidf_cols = [c for c in X_all.columns if c.startswith('tfidf_')]
struct_cols = [c for c in X_all.columns if not c.startswith('tfidf_')]

print(f'TF-IDF features:    {len(tfidf_cols)}')
print(f'Structured features: {len(struct_cols)}')
print(f'Structured feature names: {struct_cols}')

In [ ]:
# Add h-index features to structured features
X_struct_train = X_all.loc[train_idx, struct_cols].copy()
X_struct_test  = X_all.loc[test_idx,  struct_cols].copy()

if len(hindex_features.columns) > 0:
    h_train = hindex_features.loc[train_idx]
    h_test  = hindex_features.loc[test_idx]
    
    X_struct_train = pd.concat([X_struct_train, h_train], axis=1)
    X_struct_test  = pd.concat([X_struct_test,  h_test],  axis=1)
    print(f'Structured + H-index features: {X_struct_train.shape[1]}')

# Also prepare full TF-IDF baseline (existing best)
X_tfidf_train = X_all.loc[train_idx]
X_tfidf_test  = X_all.loc[test_idx]

print(f'Full feature matrix (TF-IDF + structured): {X_tfidf_train.shape}')

## 5. SciBERT Abstract Embeddings

Use `allenai/scibert_scivocab_uncased` to encode paper abstracts into dense 768-dim vectors.
Embeddings are cached to disk so this cell only runs once (takes ~10–30 min on CPU).

In [ ]:
EMBEDDING_CACHE = feature_dir / 'scibert_embeddings.pkl'
SCIBERT_MODEL   = 'allenai/scibert_scivocab_uncased'

def encode_abstracts_scibert(texts, model_name=SCIBERT_MODEL, batch_size=32, max_length=512):
    """Encode abstracts with SciBERT using mean-pooling over token embeddings."""
    import torch
    from transformers import AutoTokenizer, AutoModel
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {device}')
    print(f'Loading {model_name}...')
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    
    all_embeddings = []
    texts = [str(t) if pd.notna(t) else '' for t in texts]
    
    from tqdm import tqdm
    for i in tqdm(range(0, len(texts), batch_size), desc='Encoding'):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(device)
        
        with torch.no_grad():
            output = model(**encoded)
        
        # Mean-pool over non-padding tokens
        attention_mask = encoded['attention_mask'].unsqueeze(-1).float()
        token_embeddings = output.last_hidden_state
        mean_emb = (token_embeddings * attention_mask).sum(1) / attention_mask.sum(1)
        all_embeddings.append(mean_emb.cpu().numpy())
    
    return np.vstack(all_embeddings)


# Load from cache or compute
if EMBEDDING_CACHE.exists():
    print(f'Loading cached embeddings from {EMBEDDING_CACHE}')
    with open(EMBEDDING_CACHE, 'rb') as f:
        embedding_data = pickle.load(f)
    scibert_embeddings = embedding_data['embeddings']
    embedding_index    = embedding_data['index']
    print(f'Loaded embeddings: {scibert_embeddings.shape}')
else:
    print('Computing SciBERT embeddings (this will take 10-30 min on CPU)...')
    all_idx_ordered = list(train_idx) + list(test_idx)
    abstracts = df.loc[all_idx_ordered, 'Abstract'].tolist()
    
    scibert_embeddings = encode_abstracts_scibert(abstracts)
    embedding_index    = all_idx_ordered
    
    # Cache to disk
    feature_dir.mkdir(parents=True, exist_ok=True)
    with open(EMBEDDING_CACHE, 'wb') as f:
        pickle.dump({'embeddings': scibert_embeddings, 'index': embedding_index}, f)
    print(f'Saved to {EMBEDDING_CACHE}')
    print(f'Embeddings shape: {scibert_embeddings.shape}')

In [ ]:
# Align embeddings with train/test indices
idx_to_pos = {idx: pos for pos, idx in enumerate(embedding_index)}

train_positions = [idx_to_pos[i] for i in train_idx]
test_positions  = [idx_to_pos[i] for i in test_idx]

emb_train_raw = scibert_embeddings[train_positions]   # (n_train, 768)
emb_test_raw  = scibert_embeddings[test_positions]    # (n_test, 768)

print(f'Train embeddings: {emb_train_raw.shape}')
print(f'Test  embeddings: {emb_test_raw.shape}')

# PCA to 256 dims (reduces noise, speeds up LR training)
N_COMPONENTS = 256
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
emb_train_pca = pca.fit_transform(emb_train_raw)
emb_test_pca  = pca.transform(emb_test_raw)

explained = pca.explained_variance_ratio_.cumsum()[-1]
print(f'\nPCA {N_COMPONENTS} components explain {explained:.1%} of variance')

# Convert to DataFrames for easy concatenation
emb_cols = [f'scibert_{i}' for i in range(N_COMPONENTS)]
df_emb_train = pd.DataFrame(emb_train_pca, index=train_idx, columns=emb_cols)
df_emb_test  = pd.DataFrame(emb_test_pca,  index=test_idx,  columns=emb_cols)

## 6. Evaluate Configurations

Helper function to train and evaluate a model with threshold optimization.

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, label=''):
    """Train model, optimize threshold, and return F1/AUC."""
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    auc   = roc_auc_score(y_test, proba)
    
    # Threshold search
    best_f1, best_thr = 0, 0.5
    for thr in np.arange(0.30, 0.76, 0.01):
        preds = (proba >= thr).astype(int)
        f1 = f1_score(y_test, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    
    preds = (proba >= best_thr).astype(int)
    prec = precision_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)
    
    print(f'{label:55s}  F1={best_f1:.4f}  AUC={auc:.4f}  P={prec:.4f}  R={rec:.4f}  thr={best_thr:.2f}')
    return {'label': label, 'f1': best_f1, 'auc': auc, 'precision': prec, 'recall': rec, 'threshold': best_thr}

results = []

### Config 0 — Baseline (current best: TF-IDF 5k + Venue + Author, no H-index)

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
r = evaluate_model(lr, X_tfidf_train.fillna(0), X_tfidf_test.fillna(0), y_train, y_test,
                   label='Baseline: TF-IDF 5k + Venue + Author (LR)')
results.append(r)

### Config A — Venue + Author + H-Index (no text)

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
r = evaluate_model(lr, X_struct_train.fillna(0), X_struct_test.fillna(0), y_train, y_test,
                   label='Config A: Venue + Author + H-index only (LR)')
results.append(r)

lgbm = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                           class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)
r = evaluate_model(lgbm, X_struct_train.fillna(0), X_struct_test.fillna(0), y_train, y_test,
                   label='Config A: Venue + Author + H-index only (LightGBM)')
results.append(r)

### Config B — SciBERT + Venue + Author + H-Index

In [ ]:
X_full_train = pd.concat([df_emb_train, X_struct_train.fillna(0)], axis=1)
X_full_test  = pd.concat([df_emb_test,  X_struct_test.fillna(0)],  axis=1)

print(f'Combined feature matrix: {X_full_train.shape}')

# LogisticRegression on SciBERT + structured
lr = LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=-1,
                        C=1.0, random_state=RANDOM_STATE)
r = evaluate_model(lr, X_full_train, X_full_test, y_train, y_test,
                   label='Config B: SciBERT-PCA256 + Venue + Author + H-index (LR)')
results.append(r)

In [ ]:
# LightGBM on SciBERT + structured
lgbm = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                           class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)
r = evaluate_model(lgbm, X_full_train, X_full_test, y_train, y_test,
                   label='Config B: SciBERT-PCA256 + Venue + Author + H-index (LightGBM)')
results.append(r)

In [ ]:
# Random Forest on SciBERT + structured
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
r = evaluate_model(rf, X_full_train, X_full_test, y_train, y_test,
                   label='Config B: SciBERT-PCA256 + Venue + Author + H-index (RF)')
results.append(r)

### Config C — SciBERT Only (ablation)

In [ ]:
lr = LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=-1,
                        random_state=RANDOM_STATE)
r = evaluate_model(lr, df_emb_train, df_emb_test, y_train, y_test,
                   label='Config C: SciBERT-PCA256 only (LR)')
results.append(r)

lgbm = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                           class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)
r = evaluate_model(lgbm, df_emb_train, df_emb_test, y_train, y_test,
                   label='Config C: SciBERT-PCA256 only (LightGBM)')
results.append(r)

### Config D — Full SciBERT (768 dims, no PCA) + Structured

In [ ]:
# Try full 768-dim embeddings with LightGBM (tree-based models handle high-dim well)
emb_cols_full = [f'scibert_full_{i}' for i in range(768)]
df_emb_train_full = pd.DataFrame(emb_train_raw, index=train_idx, columns=emb_cols_full)
df_emb_test_full  = pd.DataFrame(emb_test_raw,  index=test_idx,  columns=emb_cols_full)

X_full768_train = pd.concat([df_emb_train_full, X_struct_train.fillna(0)], axis=1)
X_full768_test  = pd.concat([df_emb_test_full,  X_struct_test.fillna(0)],  axis=1)

print(f'Full 768-dim feature matrix: {X_full768_train.shape}')

lgbm = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                           class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)
r = evaluate_model(lgbm, X_full768_train, X_full768_test, y_train, y_test,
                   label='Config D: SciBERT-768 + Venue + Author + H-index (LightGBM)')
results.append(r)

### Config E — TF-IDF 5k + Venue + Author + H-Index (add h-index to existing baseline)

In [ ]:
# Add H-index to the existing TF-IDF feature matrix
if len(hindex_features.columns) > 0:
    X_tfidf_hindex_train = pd.concat([X_tfidf_train.fillna(0), hindex_features.loc[train_idx]], axis=1)
    X_tfidf_hindex_test  = pd.concat([X_tfidf_test.fillna(0),  hindex_features.loc[test_idx]],  axis=1)

    lr = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
    r = evaluate_model(lr, X_tfidf_hindex_train, X_tfidf_hindex_test, y_train, y_test,
                       label='Config E: TF-IDF 5k + Venue + Author + H-index (LR)')
    results.append(r)

    lgbm = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                               class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)
    r = evaluate_model(lgbm, X_tfidf_hindex_train, X_tfidf_hindex_test, y_train, y_test,
                       label='Config E: TF-IDF 5k + Venue + Author + H-index (LightGBM)')
    results.append(r)
else:
    print('Skipped Config E — no h-index features available')

## 7. Results Summary

In [ ]:
results_df = pd.DataFrame(results).sort_values('f1', ascending=False)

baseline_f1 = results_df[results_df['label'].str.startswith('Baseline')]['f1'].values[0]
best_prev   = 0.6333  # Exp 10c selective domain segmentation

print('=' * 100)
print('FINAL RESULTS SUMMARY — Experiment 14: SciBERT + Author H-Index')
print('=' * 100)
print(f'{"Config":<55}  {"F1":>8}  {"vs TF-IDF baseline":>20}  {"vs best-ever (63.33%)": >22}')
print('-' * 100)
for _, row in results_df.iterrows():
    delta_base = (row['f1'] - baseline_f1) * 100
    delta_best = (row['f1'] - best_prev) * 100
    marker = ' ← BEST' if row['f1'] == results_df['f1'].max() else ''
    print(f"{row['label']:<55}  {row['f1']*100:>7.2f}%  {delta_base:>+19.2f} pp  {delta_best:>+21.2f} pp{marker}")

print('=' * 100)
best = results_df.iloc[0]
print(f'\nBEST CONFIG: {best["label"]}')
print(f'  F1:        {best["f1"]*100:.2f}%')
print(f'  AUC:       {best["auc"]*100:.2f}%')
print(f'  Precision: {best["precision"]*100:.2f}%')
print(f'  Recall:    {best["recall"]*100:.2f}%')
print(f'  Threshold: {best["threshold"]:.2f}')
print(f'\n  vs TF-IDF baseline: {(best["f1"] - baseline_f1)*100:+.2f} pp')
print(f'  vs best-ever 63.33%: {(best["f1"] - best_prev)*100:+.2f} pp')